author: ransell

# TT & TS Data Loading



# Data Loading

Filtering down to the data needed for the BT Ranking implementation

This does not include the data on the likert scale, or the open ended questions

Data needed: VALUES survey data from qualtrics

In [1]:
import pandas as pd

# Name of the ID column in main file
id_column = "ResponseId"

df = pd.read_csv("VALUES-humanness_june_2025_TS_TT_July 14, 2025_17.06.csv")

columns_to_keep = [id_column]  # always keep the ID column

for col in df.columns:
    if col.startswith("tt_res_") or col.startswith("ts_res_"):
        columns_to_keep.append(col)
    elif col.startswith("tt_pairs_") or col.startswith("ts_pairs_"):
        columns_to_keep.append(col)


columns_to_keep = list(dict.fromkeys(columns_to_keep))

subset_df = df[columns_to_keep]

# 4. Read the CSV that contains failed-attention IDs
failed_attn_df = pd.read_csv("data/failed_attn_ids.csv")
failed_ids = failed_attn_df[id_column].tolist()

# Filter out rows whose ResponseId is in the failed ID list
subset_df = subset_df[~subset_df[id_column].isin(failed_ids)]


subset_df.to_csv("data/subset_data.csv", index=False)

print(subset_df.head())


          ResponseId tt_res_1 tt_res_2 tt_res_3 tt_res_4 tt_res_10 tt_res_5  \
3  R_7rxMhbT41kRwKcR        2        2        2        1         1        1   
4  R_7ey2gWQDsQiyNfs        1        1        2        2         1        1   
5  R_5e8Ak76J0LuWQl5        1        1        2        1         1        2   
6  R_5QZ064dpc7ehzBF        2        1        1        1         1        2   
8  R_1V1MgGGmtNuHcLL        1        2        1        2         1        1   

  tt_res_6 tt_res_7 tt_res_8  ... tt_pairs_1.id.0  \
3        2        1        1  ...             263   
4        1        1        1  ...             422   
5        1        2        2  ...             237   
6        1        1        2  ...             466   
8        2        2        1  ...             588   

                                   tt_pairs_1.post.0 tt_pairs_2.id.0  \
3    Trump’s ideas are just...  Kamala's the future!             400   
4  Harris's commitment to reinstating Roe v. Wade...          

Code block to get one pair per row

In [2]:
import pandas as pd

df = pd.read_csv("data/subset_data.csv")

# List to hold the long-format rows
rows = []

# Loop over every row (participant) in the dataset
for idx, row in df.iterrows():
    participant_id = row["ResponseId"]

    # Loop over all 'ts' questions 
    for i in range(1, 16):
        # Skip question #10 (attention check)
        if i == 10:
            continue

        res_col        = f"ts_res_{i}"
        left_id_col    = f"ts_pairs_{i}.left_id.0"
        left_text_col  = f"ts_pairs_{i}.left_text.0"
        right_text_col = f"ts_pairs_{i}.right_text.0"
        right_id_col   = f"ts_pairs_{i}.right_id.0"

        if res_col in df.columns:
            rows.append({
                "ResponseId":    participant_id,
                "question_type": "ts",
                "question_num":  i,
                "user_response": row.get(res_col),
                "left_id":       row.get(left_id_col),
                "left_text":     row.get(left_text_col),
                "right_text":    row.get(right_text_col),
                "right_id":      row.get(right_id_col)
            })

    # Loop over all 'tt' questions 
    for i in range(1, 16):
        # Skip question #10 (attention check)
        if i == 10:
            continue

        res_col        = f"tt_res_{i}"
        left_id_col    = f"tt_pairs_{i}.left_id.0"
        left_text_col  = f"tt_pairs_{i}.left_text.0"
        right_text_col = f"tt_pairs_{i}.right_text.0"
        right_id_col   = f"tt_pairs_{i}.right_id.0"

        if res_col in df.columns:
            rows.append({
                "ResponseId":    participant_id,
                "question_type": "tt",
                "question_num":  i,
                "user_response": row.get(res_col),
                "left_id":       row.get(left_id_col),
                "left_text":     row.get(left_text_col),
                "right_text":    row.get(right_text_col),
                "right_id":      row.get(right_id_col)
            })

# Convert the list of dictionaries into a DataFrame 
reshaped_df = pd.DataFrame(rows)

# Function to assign winner and loser based on the user's response
def assign_winner(row):
    response = str(row['user_response']).strip()  # stripped of whitespace
    if response == '1':
        return pd.Series({
            "post_winner": row["left_text"],
            "id_winner": row["left_id"],
            "post_loser": row["right_text"],
            "id_loser": row["right_id"]
        })
    elif response == '2':
        return pd.Series({
            "post_winner": row["right_text"],
            "id_winner": row["right_id"],
            "post_loser": row["left_text"],
            "id_loser": row["left_id"]
        })
    else:
        # If the response doesn't match expected values, defaults to None
        return pd.Series({
            "post_winner": None,
            "id_winner": None,
            "post_loser": None,
            "id_loser": None
        })

# Apply the function to every row to create new columns for winners and losers
winner_loser_cols = reshaped_df.apply(assign_winner, axis=1)
reshaped_df = pd.concat([reshaped_df, winner_loser_cols], axis=1)

output_csv = "data/reshaped_data_with_winner.csv"
reshaped_df.to_csv(output_csv, index=False)

print("Reshaped data with winner/loser information saved to:", output_csv)
print(reshaped_df.head())


Reshaped data with winner/loser information saved to: data/reshaped_data_with_winner.csv
          ResponseId question_type  question_num  user_response  left_id  \
0  R_7rxMhbT41kRwKcR            ts             1              2      408   
1  R_7rxMhbT41kRwKcR            ts             2              2      399   
2  R_7rxMhbT41kRwKcR            ts             3              2      101   
3  R_7rxMhbT41kRwKcR            ts             4              2      522   
4  R_7rxMhbT41kRwKcR            ts             5              1      453   

                                           left_text  \
0  it's time for a change...kamala harris 2024 💖 ...   
1  Kamala Harris's health care plan is the FUTURE...   
2  This just in from #MAGACULT LAND, the Culkin s...   
3  Why is Trump still denying the effects of his ...   
4  I’m not a fan of Trump’s style, but he did hav...   

                                          right_text  right_id  \
0  '@USER @USER @USER The Trumplicans? #President..

We now have the data seperated into TT and TS, along with the winners and losers and their IDs.

In [3]:
import pandas as pd

# bring back the DataFrame you previously saved
reshaped_df = pd.read_csv("data/reshaped_data_with_winner.csv")

# now filter as before
ts_df = reshaped_df.loc[
    reshaped_df["question_type"] == "ts",
    ["ResponseId", "post_winner", "id_winner", "post_loser", "id_loser"]
]
tt_df = reshaped_df.loc[
    reshaped_df["question_type"] == "tt",
    ["ResponseId", "post_winner", "id_winner", "post_loser", "id_loser"]
]

ts_df.to_csv("data/ts_data.csv", index=False)
tt_df.to_csv("data/tt_data.csv", index=False)

print("Saved ts questions to 'ts_data.csv' and tt questions to 'tt_data.csv'.")


Saved ts questions to 'ts_data.csv' and tt questions to 'tt_data.csv'.
